In [ ]:
from openpyxl import load_workbook

# --- Input file paths ---
source_file = r"C:\Users\LENOVO\Downloads\77335941.xlsx"   # Actual data
mapping_file = r"C:\Users\LENOVO\Downloads\Dummy file.xlsx"  # Mapping (ABI col | Source col or "fixed")
abi_template = r"C:\Users\LENOVO\Downloads\T86 to ABI Columns (1).xlsx"  # ABI Template
output_file = r"C:\Users\LENOVO\Documents\ABI_output.xlsx"

# --- Load workbooks ---
src_wb = load_workbook(source_file, data_only=True)
src_ws = src_wb.active
map_wb = load_workbook(mapping_file, data_only=True)
map_ws = map_wb.active
abi_wb = load_workbook(abi_template)
abi_ws = abi_wb.active

# --- Build mapping dictionary: ABI col -> (source col or fixed value) ---
mapping = {}
for row in map_ws.iter_rows(min_row=2, values_only=True):
    abi_col, src_or_val = row
    if abi_col and src_or_val:
        src_or_val = str(src_or_val).strip()
        mapping[abi_col.strip()] = src_or_val

print("✅ Mapping loaded:")
for k, v in mapping.items():
    print(f"ABI: {k}  <--  {v}")

# --- Extract source headers ---
src_headers_row = next(src_ws.iter_rows(min_row=1, max_row=1, values_only=True))
src_headers = {str(col).strip(): idx for idx, col in enumerate(src_headers_row, start=1)}

print("\n✅ Source headers found in source file:")
print(src_headers)

# --- Extract ABI headers row from template ---
abi_headers_row = next(abi_ws.iter_rows(min_row=1, max_row=1, values_only=True))
abi_headers = {str(col).strip(): idx for idx, col in enumerate(abi_headers_row, start=1)}

# --- Start writing data at row 5 (first record row in ABI template) ---
start_row = 5
for row_idx, row in enumerate(src_ws.iter_rows(min_row=2, values_only=True), start=start_row):
    src_row_dict = {str(col).strip(): row[idx-1] for col, idx in src_headers.items()}
    
    for abi_col, src_or_val in mapping.items():
        if abi_col in abi_headers:
            target_col_idx = abi_headers[abi_col]
            
            # Case 1: fixed value (inside quotes)
            if src_or_val.startswith('"') and src_or_val.endswith('"'):
                value = src_or_val.strip('"')
            
            # Case 2: copy from source column
            elif src_or_val in src_headers:
                value = src_row_dict.get(src_or_val, "")
            
            # Case 3: unknown (leave blank)
            else:
                value = ""
            
            abi_ws.cell(row=row_idx, column=target_col_idx, value=value)

# --- Save output ---
abi_wb.save(output_file)
print(f"\n✅ ABI file created: {output_file}")